# 🔬 Arm 5: S³-GRPO (Structural, Stepwise & Invariant Policy Optimization)
**Project:** Diagnosing and Resolving Code SLM Mimicry  
**Authors:** Omar Abdelhamid, Nour Walid  
**Supervisor:** Dr. Ghada Soliman  
**Organization:** Orange Innovation Labs — AI Research & Advanced Innovation Division  

---

## 🎯 Research Motivation & Scientific Stamp
In our empirical evaluation across 7 models (5,348 benchmark runs):
- **M4 Standard GRPO** achieved high accuracy (97.9% AUC) but suffered from **Reasoning Bloat / Overthinking Tax = 2.724** (length hacking).
- **M5 AST-RL** cut overthinking by **51.6% (Tax = 1.318)** and achieved the highest OOD generalization (10.0%).
- **M7 Step-RLVR** unlocked near-perfect multi-step accuracy (99% L1, 100% L3) via dense assertion verification.

### The Unified S³-GRPO Formulation:
$$\mathcal{R}_{\text{total}}(\hat{y}, x) = w_{\text{step}} \cdot \mathcal{R}_{\text{step}}(\hat{y}) + w_{\text{ast}} \cdot \text{sim}_{\text{AST}}(\hat{y}, y^*) - w_{\text{inv}} \cdot \mathcal{L}_{\text{inv}}(x, x') - w_{\text{tax}} \cdot \Omega_{\text{parsimony}}(\hat{y}, y^*)$$

| Component | Literature Grounding | Responsibility |
|---|---|---|
| **Stepwise PRM ($\mathcal{R}_{\text{step}}$)** | CodePRM (ACL 2025), ExecVerify (ICSE 2026) | Dense assertion credit on complex tasks |
| **AST Similarity ($\text{sim}_{\text{AST}}$)** | TreeDiff (ASE 2025), VeriSeek (ICSE 2025) | Inductive bias against syntactic rambling |
| **Invariance Loss ($\mathcal{L}_{\text{inv}}$)** | Inv-GRPO (Aly et al. 2026) | Cross-prompt surface robustness |
| **Parsimony Penalty ($\Omega_{\text{parsimony}}$)** | Our Novel Contribution | Directly penalizes token verbosity & hallucination |

In [ ]:
# ── 1. Setup Environment & Path Resolution ────────────────────────────────────
import os, sys, subprocess
from pathlib import Path

if os.path.exists('/kaggle'):
    print('Running inside Kaggle environment.')
    os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
    if os.path.exists('/kaggle/working/reo/src'):
        REPO_ROOT = '/kaggle/working/reo'
        subprocess.run(['git', '-C', REPO_ROOT, 'pull', 'origin', 'main'])
    else:
        subprocess.run(['git', 'clone', 'https://github.com/OmarAbdelhamidAly/diagnosing-resolving-code-slm-mimicry.git', '/kaggle/working/reo'])
        REPO_ROOT = '/kaggle/working/reo'
else:
    _nb_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
    REPO_ROOT = str(Path(_nb_dir).resolve().parents[0]) if Path(_nb_dir).name == 'notebooks' else str(Path(_nb_dir).resolve())

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f'Working directory set to: {os.getcwd()}')

# Check PyTorch & GPU
import torch
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device      : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)')


In [ ]:
# ── 2. Load Multi-Rung Training Pool ──────────────────────────────────────────
from src.evaluation.registry import BenchmarkRegistry

data_dir = Path(REPO_ROOT) / 'data' / 'ladder'
if not (data_dir / 'L0_humaneval_standard.jsonl').exists() and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'L0_humaneval_standard.jsonl' in files:
            data_dir = Path(root)
            break

print(f'Loading tasks from: {data_dir}')
registry = BenchmarkRegistry(data_dir=data_dir)

# Multi-rung pool with canonical solutions: L1, L3, L4, L5
tasks = [
    {
        'prompt': t.prompt,
        'test': t.test,
        'entry_point': t.entry_point,
        'canonical_solution': t.canonical_solution or '',
    }
    for t in (
        registry.get_tasks('L1')
        + registry.get_tasks('L3')
        + registry.get_tasks('L4')
        + registry.get_tasks('L5')
    )
]
print(f'✅ Successfully loaded {len(tasks)} tasks for S³-GRPO training.')


In [ ]:
# ── 3. Dry-Run Verification of S³-GRPO Reward Engine ─────────────────────────
from src.arms.arm5_hybrid_s3.reward_engine import S3RewardEngine

engine = S3RewardEngine(
    w_step=0.50,
    w_ast=0.30,
    w_inv=0.20,
    w_parsimony=0.15,
    sandbox_timeout=3.0,
)

sample_task = tasks[0]
print('Evaluating sample task with reference solution:')
print('Prompt sample:', sample_task['prompt'][:100], '...')

res = engine.compute_reward(
    prompt=sample_task['prompt'],
    solution=sample_task['canonical_solution'],
    test=sample_task['test'],
    entry_point=sample_task['entry_point'],
    canonical_solution=sample_task['canonical_solution'],
)
print('Dry-run Reward Decomposition:', res)
assert res['all_passed'] is True
print('✅ Reward Engine Verified: All assertions passed cleanly.')


In [ ]:
# ── 4. Execute 500-Step S³-GRPO Training Loop ────────────────────────────────
from src.arms.arm5_hybrid_s3.trainer import S3GRPOTrainer

OUTPUT_DIR = 'checkpoints/s3_grpo_final'
trainer = S3GRPOTrainer(
    output_dir=OUTPUT_DIR,
    group_size=4,
    learning_rate=1e-5,
    max_new_tokens=384,
    w_step=0.50,
    w_ast=0.30,
    w_inv=0.20,
    w_parsimony=0.15,
)

NUM_STEPS = 500
print(f'🚀 Starting S³-GRPO Optimization ({NUM_STEPS} steps)...')
history = trainer.train(
    tasks=tasks,
    num_steps=NUM_STEPS,
    grad_accum_steps=2,
)
print('🎉 Training Completed Successfully! Checkpoint saved to:', OUTPUT_DIR)


In [ ]:
# ── 5. Plot S³-GRPO Training Dynamics ─────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history['step'], history['loss'], color='#E74C3C', lw=1.5)
axes[0].set_title('Policy Loss (Surrogate Objective)')
axes[0].set_xlabel('Step')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['step'], history['mean_reward'], color='#2ECC71', lw=1.5, label='Composite R')
axes[1].plot(history['step'], history['mean_step_reward'], color='#3498DB', lw=1.2, ls='--', label='Stepwise R')
axes[1].set_title('Composite & Process Rewards')
axes[1].set_xlabel('Step')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(history['step'], history['mean_ast_sim'], color='#9B59B6', lw=1.5, label='AST Tree Sim')
axes[2].plot(history['step'], history['mean_parsimony_penalty'], color='#F39C12', lw=1.2, ls=':', label='Parsimony Tax')
axes[2].set_title('Syntactic & Parsimony Dynamics')
axes[2].set_xlabel('Step')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/s3_grpo_training_dynamics.png', dpi=300)
plt.show()
print('💾 Saved training dynamics plot to results/s3_grpo_training_dynamics.png')
